In [1]:
import cv2
import numpy as np
import torch
import pyttsx3
import tkinter as tk
from tkinter import ttk
import speech_recognition as sr
from threading import Thread
from ultralytics import YOLO
import yaml
import time

class VisionAssistant:
    def __init__(self):
        # Initialize variables
        self.detection_running = False
        self.stop_detection_flag = False
        self.stop_voice_thread = False
        self.last_spoken = ""
        self.confidence_threshold = 0.25
        
        # Load models
        self.load_models()
        
        # Setup TTS
        self.setup_tts()
        
        # Setup GUI
        self.setup_gui()
        
        # Start voice command listener
        self.start_voice_listener()
    
    def load_models(self):
        """Load YOLO and MiDaS models"""
        print("Loading models...")
        
        # Load YOLO model
        self.yolo_model = YOLO(r"C:\Users\User\Downloads\best (1).pt")
        
        # Load class names
        try:
            with open(r"C:\Users\User\open-images-yolo\data.yaml", "r") as stream:
                data = yaml.safe_load(stream)
                self.class_names = data["names"]
        except FileNotFoundError:
            # Fallback class names
            self.class_names = ['Person', 'Bottle', 'Chair', 'Backpack', 'Laptop', 
                              'Book', 'Mobile phone', 'Television', 'Table', 
                              'Computer keyboard', 'Car', 'Couch', 'Bench', 
                              'Waste container', 'Dog']
        
        # Load MiDaS model for depth estimation
        device = "cpu"
        model_type = "DPT_Large"
        self.midas_model = torch.hub.load("intel-isl/MiDaS", model_type)
        self.midas_model.to(device)
        self.midas_model.eval()
        
        # Load transforms
        midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
        if model_type in ("DPT_Large", "DPT_Hybrid"):
            self.transform = midas_transforms.dpt_transform
        else:
            self.transform = midas_transforms.small_transform
            
        print("Models loaded successfully!")
    
    def setup_tts(self):
        """Setup text-to-speech engine"""
        self.engine = pyttsx3.init()
        self.engine.setProperty('rate', 150)
        self.engine.setProperty('volume', 0.9)
    
    def get_basic_color_name(self, rgb):
        """Get basic color name from RGB values"""
        colors = {
            'red': (255, 0, 0), 'green': (0, 255, 0), 'blue': (0, 0, 255),
            'yellow': (255, 255, 0), 'orange': (255, 165, 0), 'purple': (128, 0, 128),
            'pink': (255, 192, 203), 'brown': (150, 75, 0), 'black': (0, 0, 0),
            'white': (255, 255, 255), 'gray': (128, 128, 128)
        }
        min_dist = float('inf')
        closest = "unknown"
        for name, c_rgb in colors.items():
            dist = np.linalg.norm(np.array(rgb) - np.array(c_rgb))
            if dist < min_dist:
                min_dist = dist
                closest = name
        return closest
    
    def run_detection(self):
        """Main detection loop"""
        cap = cv2.VideoCapture(0)
        self.last_spoken = ""

        while self.detection_running and not self.stop_detection_flag:
            ret, frame = cap.read()
            if not ret:
                break

            # MiDaS depth estimation
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            input_batch = self.transform(img_rgb).to("cpu")
            if input_batch.ndim == 3:
                input_batch = input_batch.unsqueeze(0)
            
            with torch.no_grad():
                prediction = self.midas_model(input_batch)
                depth_map = prediction.squeeze().cpu().numpy()

            # Normalize depth map for better distance estimation
            norm_depth = (depth_map - np.min(depth_map)) / (np.max(depth_map) - np.min(depth_map) + 1e-6)

            # YOLO object detection
            results = self.yolo_model(frame, verbose=False)  # Suppress YOLO output
            objects_in_frame = []

            # Check if any detections exist
            if results[0].boxes is not None and len(results[0].boxes) > 0:
                for box in results[0].boxes:
                    conf = float(box.conf[0])
                    if conf < self.confidence_threshold:
                        continue

                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cls_id = int(box.cls[0])
                    label = self.class_names[cls_id] if cls_id < len(self.class_names) else f'class_{cls_id}'

                    # Ensure valid bounding box
                    if x2 <= x1 or y2 <= y1:
                        continue

                    # Extract object crop for color analysis
                    obj_crop = frame[y1:y2, x1:x2]
                    if obj_crop.size == 0:
                        continue

                    # Get dominant color from center region
                    h, w = obj_crop.shape[:2]
                    if h > 10 and w > 10:  # Ensure crop is large enough
                        cx1, cx2 = max(0, int(w * 0.3)), min(w, int(w * 0.7))
                        cy1, cy2 = max(0, int(h * 0.3)), min(h, int(h * 0.7))

                        obj_crop_rgb = cv2.cvtColor(obj_crop, cv2.COLOR_BGR2RGB)
                        center_crop_rgb = obj_crop_rgb[cy1:cy2, cx1:cx2]
                        
                        if center_crop_rgb.size > 0:
                            # Get average color more robustly
                            avg_color = np.mean(center_crop_rgb.reshape(-1, 3), axis=0)
                            color_name = self.get_basic_color_name(tuple(avg_color.astype(int)))
                        else:
                            color_name = "unknown"
                    else:
                        color_name = "unknown"

                    # Distance calculation with improved bounds checking
                    center_x = max(0, min((x1 + x2) // 2, depth_map.shape[1] - 1))
                    center_y = max(0, min((y1 + y2) // 2, depth_map.shape[0] - 1))
                    
                    # Sample area around center for better depth estimation
                    depth_values = []
                    for dy in range(-2, 3):
                        for dx in range(-2, 3):
                            ny = max(0, min(center_y + dy, depth_map.shape[0] - 1))
                            nx = max(0, min(center_x + dx, depth_map.shape[1] - 1))
                            depth_values.append(depth_map[ny, nx])
                    
                    depth_val = np.median(depth_values)  # Use median for stability
                    
                    # Improved distance scaling
                    DEPTH_SCALE_CM = 1  # Adjusted for better estimates
                    approx_dist_cm = round(abs(depth_val * DEPTH_SCALE_CM), 2)
                    approx_dist_ft = round(approx_dist_cm / 30.48, 2)

                    # Height estimation with better scaling
                    box_height_pixels = y2 - y1
                    PIXEL_TO_CM_RATIO = 0.07
                    approx_height_cm = round(box_height_pixels * PIXEL_TO_CM_RATIO, 0)

                    # Format strings
                    height_str = f"{int(approx_height_cm)}"
                    dist_cm_str = f"{approx_dist_cm:.2f}"
                    dist_ft_str = f"{approx_dist_ft:.2f}"

                    # Generate description in the exact format requested
                    if label.lower() == "person":
                        desc = f"A person, about {height_str} cm tall, is {dist_ft_str} feet ({dist_cm_str} cm) away."
                        overlay_text = f"Person: {height_str} cm, {dist_ft_str} ft"
                    else:
                        desc = f"The {label} of {color_name} color, about {height_str} cm tall, is {dist_ft_str} feet ({dist_cm_str} cm) away."
                        overlay_text = f"{label}: {color_name}, {height_str} cm, {dist_ft_str} ft"

                    print(f"[{label}] {desc}")
                    objects_in_frame.append(desc)

                    # Draw bounding box and overlay text with better visibility
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    
                    # Add background to text for better readability
                    text_size = cv2.getTextSize(overlay_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)[0]
                    cv2.rectangle(frame, (x1, y2 + 5), (x1 + text_size[0] + 10, y2 + 25), (0, 0, 0), -1)
                    cv2.putText(frame, overlay_text, (x1 + 5, y2 + 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)
                    
                    # Add confidence score
                    conf_text = f"Conf: {conf:.2f}"
                    cv2.putText(frame, conf_text, (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

            # Text-to-Speech Output
            if objects_in_frame:
                message = " Next, ".join(objects_in_frame)
                if message != self.last_spoken:
                    try:
                        # Run TTS in separate thread to avoid blocking
                        def speak_async():
                            self.engine.say(message)
                            self.engine.runAndWait()
                        Thread(target=speak_async, daemon=True).start()
                    except Exception as e:
                        print(f"TTS Error: {e}")
                    self.last_spoken = message

            # Show the frame
            cv2.imshow("Vision Assistant", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.detection_running = False
        self.stop_detection_flag = False
        self.update_status("Detection stopped.")
    
    def start_detection(self):
        """Start detection"""
        if not self.detection_running:
            self.detection_running = True
            self.stop_detection_flag = False
            Thread(target=self.run_detection, daemon=True).start()
            self.update_status("Detection running...")
            print("Detection started")
    
    def stop_detection(self):
        """Stop detection"""
        if self.detection_running:
            self.stop_detection_flag = True
            self.detection_running = False
            self.update_status("Detection stopped.")
            print("Detection stopped")
    
    def update_confidence(self, val):
        """Update confidence threshold"""
        self.confidence_threshold = float(val) / 100
        self.conf_label.config(text=f"Accuracy: {self.confidence_threshold:.2f}")
    
    def update_status(self, message):
        """Update status label"""
        if hasattr(self, 'status_label'):
            self.status_label.config(text=message)
    
    def voice_listener(self):
        """Voice command listener"""
        recognizer = sr.Recognizer()
        mic = sr.Microphone()

        try:
            with mic as source:
                recognizer.adjust_for_ambient_noise(source)
            print("Voice commands ready: 'start detection' or 'stop detection'")
        except Exception as e:
            print(f"Microphone setup error: {e}")
            return

        while not self.stop_voice_thread:
            try:
                with mic as source:
                    audio = recognizer.listen(source, phrase_time_limit=3, timeout=1)
                command = recognizer.recognize_google(audio).lower()
                print(f"Voice Command: {command}")

                if "start detection" in command and not self.detection_running:
                    self.start_detection()
                elif "stop detection" in command and self.detection_running:
                    self.stop_detection()
                    
            except sr.WaitTimeoutError:
                pass  # Continue listening
            except sr.UnknownValueError:
                pass  # Couldn't understand audio
            except Exception as e:
                print(f"Voice listener error: {e}")
                time.sleep(1)  # Brief pause on error
    
    def start_voice_listener(self):
        """Start voice command listener in background thread"""
        self.stop_voice_thread = False
        Thread(target=self.voice_listener, daemon=True).start()
    
    def setup_gui(self):
        """Setup the GUI interface matching your image"""
        self.root = tk.Tk()
        self.root.title("Vision Assistant")
        self.root.geometry("300x250")
        self.root.resizable(False, False)
        
        # Configure style
        style = ttk.Style()
        style.theme_use('clam')
        
        # Main frame
        main_frame = ttk.Frame(self.root)
        main_frame.pack(expand=True, fill='both', padx=20, pady=20)
        
        # Start Detection Button
        self.start_btn = ttk.Button(main_frame, text="Start Detection", 
                                   command=self.start_detection)
        self.start_btn.pack(pady=(0, 10), fill='x')
        
        # Stop Detection Button  
        self.stop_btn = ttk.Button(main_frame, text="Stop Detection",
                                  command=self.stop_detection)
        self.stop_btn.pack(pady=(0, 20), fill='x')
        
        # Status Label
        self.status_label = ttk.Label(main_frame, text="Ready to start detection",
                                     font=('Arial', 10))
        self.status_label.pack(pady=(0, 20))
        
        # Accuracy Slider
        slider_frame = ttk.Frame(main_frame)
        slider_frame.pack(fill='x', pady=(0, 10))
        
        self.conf_slider = ttk.Scale(slider_frame, from_=1, to=100, 
                                    orient=tk.HORIZONTAL, 
                                    command=self.update_confidence)
        self.conf_slider.set(int(self.confidence_threshold * 100))
        self.conf_slider.pack(fill='x')
        
        # Accuracy Label
        self.conf_label = ttk.Label(main_frame, 
                                   text=f"Accuracy: {self.confidence_threshold:.2f}",
                                   font=('Arial', 10))
        self.conf_label.pack()
        
        # Instructions
        instructions = ttk.Label(main_frame, 
                               text="Use buttons or voice commands:\n'start detection' or 'stop detection'",
                               font=('Arial', 8), foreground='gray')
        instructions.pack(pady=(20, 0))
        
        # Handle window closing
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
    
    def on_closing(self):
        """Handle window closing"""
        self.stop_voice_thread = True
        self.stop_detection()
        self.root.destroy()
    
    def run(self):
        """Start the application"""
        print("Vision Assistant Starting...")
        self.root.mainloop()

# Entry Point
if __name__ == "__main__":
    try:
        app = VisionAssistant()
        app.run()
    except Exception as e:
        print(f"Error starting application: {e}")
        import traceback
        traceback.print_exc()

Loading models...


Using cache found in C:\Users\User/.cache\torch\hub\intel-isl_MiDaS_master
C:\Python312\Lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Using cache found in C:\Users\User/.cache\torch\hub\intel-isl_MiDaS_master


Models loaded successfully!


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Python312\Lib\tkinter\__init__.py", line 1948, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\User\AppData\Local\Temp\ipykernel_8100\1866975459.py", line 256, in update_confidence
    self.conf_label.config(text=f"Accuracy: {self.confidence_threshold:.2f}")
    ^^^^^^^^^^^^^^^
AttributeError: 'VisionAssistant' object has no attribute 'conf_label'. Did you mean: 'conf_slider'?


Vision Assistant Starting...
Voice commands ready: 'start detection' or 'stop detection'
Detection started
[Mobile phone] The Mobile phone of gray color, about 19 cm tall, is 0.40 feet (12.30 cm) away.
Detection stopped
Detection started
Detection stopped
Voice listener error: [WinError 10054] An existing connection was forcibly closed by the remote host
